In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import jax
import jax.numpy as jnp
from utils.rep_models import EnsembleStateMetric
from flax import nnx
import optax

In [3]:
key = jax.random.PRNGKey(0)
key, sk1, sk2 = jax.random.split(key, 3)

****************************************************************************
* hwloc 2.0.3 received invalid information from the operating system.
*
* Failed with: intersection without inclusion
* while inserting Group0 (P#16 cpuset 0x00000f00) at L1d (P#16 cpuset 0x00000104)
* coming from: linux:sysfs:cluster
*
* The following FAQ entry in the hwloc documentation may help:
*   What should I do when hwloc reports "operating system" warnings?
* Otherwise please report this error message to the hwloc user's mailing list,
* along with the files generated by the hwloc-gather-topology script.
* 
* hwloc will now ignore this invalid topology information and continue.
****************************************************************************


In [4]:
g = EnsembleStateMetric(nnx.Rngs(0), 16)

In [5]:
def state_diff_function(s:jnp.ndarray, x:jnp.ndarray):
    g_sx , g_xs = g(s, x)
    u = jnp.maximum(g_sx, g_xs)[:, None]
    return jnp.mean(u)

In [6]:
s, x = jax.random.uniform(sk1, (32, 16)), jax.random.uniform(sk2, (32, 16))

In [7]:
grad_s = jax.grad(
    lambda s, x: jnp.mean(state_diff_function(s, x)),
    argnums=0
)(s, x)

In [8]:
grad_s.shape

(32, 16)

In [9]:
class SimpleNet(nnx.Module):
    def __init__(self, rngs:nnx.Rngs):

        self.trunk  = nnx.Sequential(nnx.Linear(2, 2, rngs = rngs),nnx.Linear(2, 2, rngs = rngs))

        self.head = nnx.Linear(2, 1, rngs = rngs)

    def t(self, x):
        return self.trunk(x)

    def h(self, x):
        return self.head(self.t(x))

In [10]:
nn = SimpleNet(rngs = nnx.Rngs(0))

In [11]:
x = jax.random.uniform(sk1, (10, 2))

In [12]:
def loss_f1(net:SimpleNet):
    return jnp.mean((net.t(x)-jnp.array([2, 2]*5)[:, None])**2)

def loss_f2(net:SimpleNet):
    return jnp.mean((net.h(x)-jnp.array([1, 1]*5)[:, None])**2)

In [13]:
nn_opt = nnx.Optimizer(model = nn, tx = optax.chain(optax.adam(learning_rate=1e-3)), wrt = nnx.Param)

In [14]:
f1_loss, grads1 = nnx.value_and_grad(loss_f1)(nn)
f2_loss, grads2 = nnx.value_and_grad(loss_f2)(nn)

In [21]:
nn.trunk.layers[0].kernel.get_value()

Array([[ 0.75672996, -0.6857006 ],
       [-0.5688339 , -0.8757607 ]], dtype=float32)

In [108]:
print(nn.trunk.bias, nn.head.bias)

Param( # 2 (8 B)
  value=Array([0., 0.], dtype=float32)
) Param( # 1 (4 B)
  value=Array([0.], dtype=float32)
)


In [22]:
grads1

State({
  'head': {
    'bias': Param( # 1 (4 B)
      value=Array([0.], dtype=float32)
    ),
    'kernel': Param( # 2 (8 B)
      value=Array([[0.],
             [0.]], dtype=float32)
    )
  },
  'trunk': {
    'layers': {
      0: {
        'bias': Param( # 2 (8 B)
          value=Array([-3.6091616,  1.6369729], dtype=float32)
        ),
        'kernel': Param( # 4 (16 B)
          value=Array([[-0.7365284 ,  0.33262318],
                 [-1.5688127 ,  0.70836234]], dtype=float32)
        )
      },
      1: {
        'bias': Param( # 2 (8 B)
          value=Array([-1.911083 , -1.7696725], dtype=float32)
        ),
        'kernel': Param( # 4 (16 B)
          value=Array([[0.18031734, 0.16148528],
                 [1.010051  , 0.9067663 ]], dtype=float32)
        )
      }
    }
  }
})

In [23]:
grads2

State({
  'head': {
    'bias': Param( # 1 (4 B)
      value=Array([-1.9224586], dtype=float32)
    ),
    'kernel': Param( # 2 (8 B)
      value=Array([[-0.18554366],
             [-0.4515031 ]], dtype=float32)
    )
  },
  'trunk': {
    'layers': {
      0: {
        'bias': Param( # 2 (8 B)
          value=Array([0.52570385, 0.06107386], dtype=float32)
        ),
        'kernel': Param( # 4 (16 B)
          value=Array([[0.11699163, 0.01359155],
                 [0.21856989, 0.02539245]], dtype=float32)
        )
      },
      1: {
        'bias': Param( # 2 (8 B)
          value=Array([ 1.4781656 , -0.89424413], dtype=float32)
        ),
        'kernel': Param( # 4 (16 B)
          value=Array([[-0.10065879,  0.06089543],
                 [-0.76378226,  0.46206445]], dtype=float32)
        )
      }
    }
  }
})

In [24]:
before_trunk = nn.trunk.layers[0].kernel.get_value(), nn.trunk.layers[0].bias.get_value()
before_head = nn.head.kernel.get_value(), nn.head.bias.get_value()

In [25]:
before_trunk, before_head

((Array([[ 0.75672996, -0.6857006 ],
         [-0.5688339 , -0.8757607 ]], dtype=float32),
  Array([0., 0.], dtype=float32)),
 (Array([[-0.76889336],
         [ 0.46515656]], dtype=float32),
  Array([0.], dtype=float32)))

In [26]:
# grads2.trunk.kernel.set_value(jnp.zeros_like(grads2.trunk.kernel.get_value()))
# grads2.trunk.bias.set_value(jnp.zeros_like(grads2.trunk.bias.get_value()))

grads2.trunk = jax.tree.map(
    jnp.zeros_like,
    grads2.trunk,
)

In [27]:
nn_opt.update(nn, grads2)

State({
  'head': {
    'bias': Array([0.00099999], dtype=float32),
    'kernel': Array([[0.00099999],
           [0.00099999]], dtype=float32)
  },
  'trunk': {
    'layers': {
      0: {
        'bias': Array([-0., -0.], dtype=float32),
        'kernel': Array([[-0., -0.],
               [-0., -0.]], dtype=float32)
      },
      1: {
        'bias': Array([-0., -0.], dtype=float32),
        'kernel': Array([[-0., -0.],
               [-0., -0.]], dtype=float32)
      }
    }
  }
})

In [28]:
nn.trunk.layers[0].kernel.get_value()-before_trunk[0]

Array([[0., 0.],
       [0., 0.]], dtype=float32)

In [29]:
nn.trunk.layers[0].bias.get_value()-before_trunk[1]

Array([0., 0.], dtype=float32)

In [30]:
nn.head.kernel.get_value()-before_head[0]

Array([[0.00099999],
       [0.00099999]], dtype=float32)

In [31]:
nn.head.bias.get_value()-before_head[1]

Array([0.00099999], dtype=float32)